In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")
dbutils.widgets.text("load_type", "FULL")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")
load_type = dbutils.widgets.get("load_type")

print("=" * 60)
print("STEP 1: Widget Parameters Received")
print("=" * 60)
print(f"  source_system      : {source_system}")
print(f"  table_name         : {table_name}")
print(f"  landing_timestamp  : {landing_timestamp}")
print(f"  source_file_format : {source_file_format}")
print(f"  load_type          : {load_type}")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 2: Listing Landing Volume Contents")
print("=" * 60)
landing_files = dbutils.fs.ls('/Volumes/caresync_databricks_workspace/landing/landing_volume/')
for f in landing_files:
    print(f"  {f.name:<40} size={f.size:>10,} bytes")
print(f"  Total items: {len(landing_files)}")
print("=" * 60)

In [0]:
source_path = f"/Volumes/caresync_databricks_workspace/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Guard: check if the landing folder exists and has files before reading
try:
    landing_contents = [f for f in dbutils.fs.ls(source_path) if not f.name.startswith(("_", "."))]
except Exception as e:
    if "FileNotFoundException" in str(e) or "PATH_NOT_FOUND" in str(e) or "does not exist" in str(e):
        landing_contents = []
    else:
        raise

if not landing_contents:
    print(f"No new files at {source_path} — nothing to load, skipping.")
    dbutils.notebook.exit("NO_NEW_DATA")

# Determine the file format from the widget parameter
file_format = source_file_format.strip().lower() if source_file_format else "parquet"

# Read data based on the source file format
if file_format == "parquet":
    df = spark.read.format("parquet").load(source_path)
elif file_format == "csv":
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(source_path)
elif file_format == "json":
    df = spark.read.format("json").option("multiLine", "true").load(source_path)
elif file_format == "avro":
    df = spark.read.format("avro").load(source_path)
elif file_format == "orc":
    df = spark.read.format("orc").load(source_path)
elif file_format in ("xml",):
    df = spark.read.format("xml").option("rowTag", "row").load(source_path)
elif file_format in ("delta",):
    df = spark.read.format("delta").load(source_path)
else:
    raise ValueError(f"Unsupported source file format: '{file_format}'. Supported formats: parquet, csv, json, avro, orc, xml, delta.")

row_count = df.count()
col_count = len(df.columns)

print("=" * 60)
print(f"STEP 3: Read {file_format.upper()} from Landing")
print("=" * 60)
print(f"  Source path        : {source_path}")
print(f"  Rows loaded        : {row_count:,}")
print(f"  Columns            : {col_count}")
print(f"  Column names       : {df.columns}")
print("-" * 60)
print("  Schema:")
df.printSchema()
print("-" * 60)
print("  Sample rows (up to 15):")
df.show(15, truncate=False)

In [0]:
display(df.limit(3))

In [0]:
from pyspark.sql import functions as F

landing_df = df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())

print("=" * 60)
print("STEP 4: Audit Columns Added")
print("=" * 60)
print(f"  Added columns      : landing_timestamp (lit), insert_timestamp (current)")
print(f"  Final column count : {len(landing_df.columns)}")
print(f"  Final columns      : {landing_df.columns}")
print("-" * 60)
print("  Sample rows with audit columns (up to 10):")
landing_df.select("landing_timestamp", "insert_timestamp", *[c for c in landing_df.columns if c not in ('landing_timestamp','insert_timestamp')][:5]).show(10, truncate=False)

In [0]:
display(landing_df.limit(3))

In [0]:
target_table = f"caresync_databricks_workspace.bronze.{table_name}"

print("=" * 60)
print(f"STEP 5: Write to Bronze — {target_table}")
print("=" * 60)
write_mode = "overwrite" if load_type.upper() == "FULL" else "append"
print(f"  Load type          : {load_type}")
print(f"  Write mode         : {write_mode}")
print(f"  Rows to write      : {landing_df.count():,}")

landing_df.write.mode(write_mode).saveAsTable(target_table)

# Post-write verification
bronze_df = spark.read.table(target_table)
bronze_count = bronze_df.count()
print(f"  ✅ Append complete.")
print("-" * 60)
print(f"  Bronze table total rows after write : {bronze_count:,}")
print("  Bronze sample rows (up to 10):")
bronze_df.show(10, truncate=False)
print("=" * 60)

In [0]:
%sql
select * from caresync_databricks_workspace.bronze.hospitals

In [0]:
%sql
UPDATE caresync_databricks_workspace.bronze.hospitals
SET landing_timestamp = regexp_replace(landing_timestamp, '/', '')
WHERE landing_timestamp LIKE '%/%';